In [ ]:
import numpy as np
import rasterio
import pandas as pd
from rasterio.windows import Window, from_bounds
from dask.distributed import Client, LocalCluster

# --- Input paths ---
base = "/mnt/d/GitHub/typology/data"

paths = {
    'secveg': f"{base}/sites_selection_data/input/secveg_2024_m_P.tif",
    'glad_height': f"{base}/sites_selection_data/input/GLAD_Canopy_Height_SaoPaulo_250km_P.tif",
    'veg_age': f"{base}/sites_selection_data/input/MB_c9_secFveg_age_2023_P.tif",
    'typology': f"{base}/sites_selection_data/input/20-24_combined_rc.tif",
    'patch_area': f"{base}/sites_selection_data/input/2024_area_1km.tif",
    'patch_edge': f"{base}/sites_selection_data/input/2024_edge_1km.tif",
    'patch_num': f"{base}/sites_selection_data/input/2024_pn_1km.tif",
    'shdi': f"{base}/sites_selection_data/input/brazil_coverage_2024_P_C_C.tif"
}

# --- Output paths ---
out_class_path = f"{base}/sites_selection_data/output/forest_typology_class.tif"
out_table_path = f"{base}/sites_selection_data/output/forest_typology_table.csv"

In [ ]:
# --- Initialize Dask cluster ---
cluster = LocalCluster(n_workers=2, threads_per_worker=2, memory_limit='20GB')
client = Client(cluster)
print(f"Dashboard: {client.dashboard_link}")

In [ ]:
# --- Get processing extent from typology (smallest raster) ---
with rasterio.open(paths['typology']) as ref:
    ref_bounds = ref.bounds
    ref_transform = ref.transform
    ref_crs = ref.crs
    ref_height = ref.height
    ref_width = ref.width
    ref_profile = ref.profile.copy()
    ref_nodata = ref.nodata

print(f"Processing extent: {ref_bounds}")
print(f"Dimensions: {ref_height} rows x {ref_width} cols")
print(f"Typology NoData value: {ref_nodata}")

# --- Output profile based on typology extent ---
ref_profile.update(dtype='uint8', nodata=0, count=1)

# Create output raster
with rasterio.open(out_class_path, "w", **ref_profile) as dst:
    pass

# --- Define processing function for each chunk ---
def process_chunk(row_start, row_end, ref_width, ref_transform, ref_nodata, paths):
    import numpy as np
    import rasterio
    import pandas as pd
    from rasterio.windows import Window, from_bounds

    chunk_height = row_end - row_start
    out_window = Window(0, row_start, ref_width, chunk_height)
    chunk_transform = rasterio.windows.transform(out_window, ref_transform)
    chunk_bounds = rasterio.transform.array_bounds(chunk_height, ref_width, chunk_transform)

    # Load chunk from all rasters
    chunk = {}
    for name, path in paths.items():
        with rasterio.open(path) as src:
            win = from_bounds(*chunk_bounds, transform=src.transform)
            win = win.round_offsets().round_lengths()
            data = src.read(1, window=win)
            if data.shape != (chunk_height, ref_width):
                tmp = np.zeros((chunk_height, ref_width), dtype=data.dtype)
                min_h = min(data.shape[0], chunk_height)
                min_w = min(data.shape[1], ref_width)
                tmp[:min_h, :min_w] = data[:min_h, :min_w]
                data = tmp
            chunk[name] = data

    # Mask: only process where typology has valid data
    if ref_nodata is not None:
        valid_mask = chunk['typology'] != ref_nodata
    else:
        valid_mask = ~np.isnan(chunk['typology'].astype(float))

    # Skip chunk entirely if no valid pixels
    if not valid_mask.any():
        return None, None

    # Reclassify age: 255 -> 0
    chunk['veg_age'] = np.where(chunk['veg_age'] == 255, 0, chunk['veg_age'])

    # Classification (only where typology is valid)
    result = np.zeros((chunk_height, ref_width), dtype=np.uint8)

    # 1. Mature: primary veg (class 2) + height >= 21m + age >= 30
    mature = valid_mask & (chunk['secveg'] == 2) & (chunk['glad_height'] >= 21) & (chunk['veg_age'] >= 30)
    result[mature] = 1

    # 2. Natural Regen: secondary regrowth (class 5) + age >= 5
    natural_regen = valid_mask & (chunk['secveg'] == 5) & (chunk['veg_age'] >= 5) & (result == 0)
    result[natural_regen] = 4

    # 3. Non-restored: deforestation of primary (class 4) or secondary (class 6)
    non_restored = valid_mask & ((chunk['secveg'] == 4) | (chunk['secveg'] == 6)) & (result == 0)
    result[non_restored] = 2

    # Extract table rows for classified pixels
    rows_idx, cols_idx = np.where(result > 0)
    df = None
    if len(rows_idx) > 0:
        global_rows = rows_idx + row_start
        xs, ys = rasterio.transform.xy(ref_transform, global_rows, cols_idx)

        df = pd.DataFrame({
            'x': xs,
            'y': ys,
            'class': result[rows_idx, cols_idx],
            'age': chunk['veg_age'][rows_idx, cols_idx],
            'glad_height': chunk['glad_height'][rows_idx, cols_idx],
            'secveg_class': chunk['secveg'][rows_idx, cols_idx],
            'typology': chunk['typology'][rows_idx, cols_idx],
            'patch_area': chunk['patch_area'][rows_idx, cols_idx],
            'patch_edge': chunk['patch_edge'][rows_idx, cols_idx],
            'patch_number': chunk['patch_num'][rows_idx, cols_idx],
            'shdi': chunk['shdi'][rows_idx, cols_idx]
        })

    return result, df

# --- Submit chunks to Dask workers ---
chunk_size = 2048
futures = []

for row_start in range(0, ref_height, chunk_size):
    row_end = min(row_start + chunk_size, ref_height)
    future = client.submit(
        process_chunk,
        row_start, row_end, ref_width, ref_transform, ref_nodata, paths
    )
    futures.append((row_start, row_end, future))

print(f"Submitted {len(futures)} chunks to Dask cluster...")

# --- Collect results and write output ---
all_rows = []

for row_start, row_end, future in futures:
    result, df = future.result()

    if result is not None:
        # Write classified chunk to output raster
        out_window = Window(0, row_start, ref_width, row_end - row_start)
        with rasterio.open(out_class_path, "r+") as dst:
            dst.write(result, 1, window=out_window)

    if df is not None:
        all_rows.append(df)

    print(f"  Collected rows {row_start}-{row_end} / {ref_height}")

# --- Combine and save CSV ---
print("Combining table...")
df = pd.concat(all_rows, ignore_index=True)
df.to_csv(out_table_path, index=False)

print(f"
Done!")
print(f"Classified raster: {out_class_path}")
print(f"Table exported: {out_table_path}")
print(f"Total classified pixels: {len(df)}")
print(f"  Mature (1): {(df['class'] == 1).sum()}")
print(f"  Non-restored (2): {(df['class'] == 2).sum()}")
print(f"  Natural Regen (4): {(df['class'] == 4).sum()}")

# --- Clean up ---
client.close()
cluster.close()